# Incident triage and monitoring anomalies

Review the incident classifier and the monitoring anomaly detector. Start by
comparing the two sources: incident KPIs do not always match the monitoring
readings at the same time.

Run `.venv/bin/python scripts/train_detection.py --data-dir data/raw` from the
repository root first. Use the project kernel, then run the cells in order.
Generated reports and models stay outside version control.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from banking_observability.forecasting import file_sha256
from banking_observability.detection_common import MONITORING_METRICS, metric_frame, read_incidents
from banking_observability.incident_triage import IncidentTriage, classification_metrics
from banking_observability.anomaly_detection import MonitoringDetector, anomaly_features, alert_summary

DATA = ROOT / "data/raw"
REPORTS = ROOT / "reports/detection"
MODELS = ROOT / "models/detection"
summary = json.loads((REPORTS / "summary.json").read_text())
for name, expected in summary["source_sha256"].items():
    assert file_sha256(DATA / name) == expected, f"Source changed: {name}"
incidents = read_incidents(DATA / "incident_reports.csv")
train = metric_frame(pd.read_csv(DATA / "infrastructure_metrics_train.csv"), MONITORING_METRICS, step_minutes=1)
test = metric_frame(pd.read_csv(DATA / "infrastructure_metrics_test.csv"), MONITORING_METRICS, step_minutes=1)
history = pd.concat([train, test], ignore_index=True)
cutoff = train.timestamp.iloc[-1]
triage = IncidentTriage(MODELS / "incident_triage")
anomaly = MonitoringDetector(MODELS / "monitoring_anomaly")
print(f"Incident records: {len(incidents)}; critical labels: {incidents.is_critical.sum()}")
print(f"Monitoring evaluation begins: {test.timestamp.iloc[0]}")

## Incident data consistency

Compare the incident KPIs with the monitoring readings at exactly the same time.
The discrepancy table records both values rather than replacing either one.

In [ ]:
audit = summary["incident_audit"]
discrepancies = pd.read_csv(REPORTS / "incident_discrepancies.csv")
display(pd.Series(audit, name="source_audit"))
display(discrepancies.groupby("metric").agg(records=("incident_id", "nunique"), mean_difference=("difference", "mean")))
display(discrepancies.head(5))

## Model selection and later records

Each candidate uses class weights calculated from its own training block. The
model and score threshold are chosen using chronological validation. F2 gives
more weight to recall than precision.

Then score the original monitoring readings with the same model, keeping the
incident labels fixed. This checks how the source differences affect the result;
it is not a second model for deployment.

In [ ]:
display(pd.read_csv(REPORTS / "triage_folds.csv"))
display(pd.read_csv(REPORTS / "triage_cv_leaderboard.csv").head(8))
metrics = pd.read_csv(REPORTS / "triage_test_metrics.csv")
predictions = pd.read_csv(REPORTS / "triage_test_predictions.csv", parse_dates=["timestamp"])
for comparison, rows in predictions.groupby("comparison"):
    actual = incidents.set_index("timestamp").is_critical.reindex(rows.timestamp).to_numpy()
    np.testing.assert_array_equal(actual, rows.actual)
    recalculated = classification_metrics(actual, rows.critical_score, rows.threshold.iloc[0])
    stored = metrics.loc[metrics.comparison.eq(comparison)].iloc[0]
    for name, value in recalculated.items():
        if value is not None:
            assert np.isclose(stored[name], value), (comparison, name)
display(metrics[["comparison", "n", "critical_count", "precision", "recall", "f1", "average_precision", "tn", "fp", "fn", "tp"]])

## Monitoring review threshold

Compare how consistently the forest ranks readings across different seeds. Choose
its settings on that comparison, then set the threshold on the later calibration
block. Neither step uses incident labels. The one-percent setting controls how
many calibration readings to review, not how often failures occur.

Features that never changed during fitting get a separate change check. The uptime
drop falls in calibration, so it is not counted as a test-period detection.

In [ ]:
display(pd.read_csv(REPORTS / "anomaly_stability.csv"))
features = anomaly_features(history)
evaluation_features = features.loc[features.index > cutoff]
recomputed = anomaly.model.assess(evaluation_features)
saved = pd.read_csv(REPORTS / "anomaly_test_scores.csv", parse_dates=["timestamp"])
np.testing.assert_allclose(recomputed.anomaly_score, saved.anomaly_score, atol=1e-12, rtol=0)
np.testing.assert_array_equal(recomputed.review_required, saved.review_required)
assert alert_summary(recomputed) == summary["anomaly"]["test"]
display(pd.DataFrame({period: summary["anomaly"][period] for period in ("calibration", "test")}).T)
calibration = pd.read_csv(REPORTS / "anomaly_calibration.csv")
display(calibration.loc[calibration.constant_feature_change])
display(saved.loc[saved.review_required, ["timestamp", "anomaly_score", "threshold", "application_error_rate", "auth_success_rate", "service_uptime"]])

## Inspect saved-model responses

Score the first evaluation incident and the highest-scoring monitoring minute in
the evaluation period. That minute is picked after seeing the scores to inspect
an example, not to demonstrate advance warning.

The triage score is not a calibrated probability. A monitoring flag identifies a
reading to investigate; it does not confirm an incident or its cause.

In [ ]:
incident = incidents.loc[incidents.timestamp.gt(cutoff)].iloc[0].to_dict()
display(pd.Series(triage.predict(incident), name="incident_triage"))
origin = saved.loc[saved.anomaly_score.idxmax(), "timestamp"]
response = anomaly.predict(history, as_of=str(origin))
prefix_response = anomaly.predict(history.loc[history.timestamp.le(origin)], as_of=str(origin))
assert response == prefix_response
display(pd.Series({key: value for key, value in response.items() if key not in ("largest_reference_deviations", "observed_metrics")}))
display(pd.DataFrame(response["largest_reference_deviations"]))
display(pd.Series(response["observed_metrics"], name="observed_monitoring"))

## Conclusion

Check whether the incident result holds when its KPIs are replaced by the original
monitoring values. Then inspect the flagged monitoring minutes. A high incident
classification score or an anomaly flag alone does not show that the system can
predict future incidents.

The generated report contains the counts and time splits for this run.
`docs/detection.md` explains the model settings.